# 2026/8/19

### 配环境过程

> pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu132

> pip install transformers datasets trl peft accelerate sentencepiece jupyter ipykernel trackio huggingface_hub

环境所需包都在同目录下的 sft_env.yaml 中

路径：```llm_study\week4\notes\sft_env.yaml```

In [12]:
# Import 环境导入
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

if torch.cuda.is_available():
    device = "cuda"
    print(f"Using CUDA GPU: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple MPS")
else:
    device = "cpu"
    print("Using CPU - you will need to use a GPU to train models")


Using CUDA GPU: NVIDIA GeForce RTX 5060 Ti
GPU memory: 17.1GB


由于显卡算力有限，本次使用 Qwen2.5-0.5B-Instruct 进行实验

模型链接：https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct

模型介绍：

The instruction-tuned 0.5B Qwen2.5 model, which has the following features:

- Type: Causal Language Models

- Training Stage: Pretraining & Post-training

- Architecture: transformers with RoPE, SwiGLU, RMSNorm, Attention QKV bias and tied word embeddings

- Number of Parameters: 0.49B

- Number of Paramaters (Non-Embedding): 0.36B

- Number of Layers: 24

- Number of Attention Heads (GQA): 14 for Q and 2 for KV

- Context Length: Full 32,768 tokens and generation 8192 tokens——模型上下文窗口（输入加输出 token 总数）为 32768，但是单次生成 token 数量只有 8192

In [ ]:
# 下载并加载模型
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name, dtype=torch.float16, device_map="auto"
)
print(model.device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

cuda:0


### 测试模型功能

根据模型自己的对话模板 (chat template)，加入用户输入的请求内容构成输入给模型的请求。输入给模型的内容是 token_id 序列，因此要使用分词器对输入 text 进行处理

分词后（tokenizer处理）的结果是一个字典，包含以下两个内容

  - input_ids：输入 token_ids 以及其所在 device

  - attention_mask 以及其所在 device
    
    - attention_mask 用来标记参与注意力计算的有效 token 的位置，1 有效，0 代表该位置是 padding 补齐的 token 不进行注意力计算


In [37]:
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": "讲讲量子力学"}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    # 这个参数用于设置是否加入 <|assistant|> 标记
    add_generation_prompt=True,
)
print("-" * 25 + "对话模板如下" + "-" * 25 + "\n")
print(text)
print("-" * 50 )

# 把分词后的结果以及结果转换成 PyTorch Tensor（pt张量）返回
inputs = tokenizer(text, return_tensors="pt").to(device)
print(inputs)

-------------------------对话模板如下-------------------------

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
讲讲量子力学<|im_end|>
<|im_start|>assistant

--------------------------------------------------
{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,  99526,  99526, 109539,
         114797, 151645,    198, 151644,  77091,    198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}


注意：```model.generate``` 方法获得 **输入prompt的token_ids** 和 **模型新生成回答的token_ids** 

因此在获得模型 outputs 回复的时候要将输入 token 截断，只保留输入 token 长度之后的字符串内容即可

In [38]:
with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        # 控制选择 token 是 按照概率分布随机选取 还是 直接选择最高概率
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
# print(outputs)

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("=" * 25 + "模型回答：" + "=" * 25)
print(answer)

=========================模型回答：=========================
量子力学是20世纪初由阿尔伯特·爱因斯坦、维尔纳·海森堡和伊莱恩·薛定谔等物理学家创立的物理学分支，它解释了微观世界的运动规律。以下是关于量子力学的一些基本概念和特点：

1. **波粒二象性**：量子力学中的一个核心观点，即微观粒子（如电子、光子）既表现出波动性又表现出粒子性质。

2. **不确定性原理**：由海森堡提出，指出不可能同时准确测量一个粒子的位置和动量（或动能）。这意味着我们只能在有限的时间内对粒子进行位置和动量的精确观测。

3. **量子叠加态**：在一个量子系统中，某些可能的状态可以同时存在于多个独立的状态之中，直到被观测到时才被“选择”为一个确定状态。

4. **量子纠缠**：两个或更多的粒子可以形成一种特殊的关联，使得它们之间的任何测量都会导致另一个粒子的状态发生变化，无论这两个粒子相隔多远。

5. **量子隧穿效应**：当电子穿越一个势垒时，如果这个势垒足够大，电子可能会穿过但不会直接通过，这称为量子隧穿效应。

6. **量子隧穿**：量子力学理论中的一种现象，指由于能量守恒引起的粒子无法通过一定大小的障碍物的现象。

7. **量子反常路径**：一些量子体系的行为与经典物理学中的类似行为相反，表现为粒子可以在不经过特定路径的情况下发生改变其状态，这被称为量子反常路径。

8. **量子隧穿实验**：最早使用量子隧穿现象来证明量子力学正确的实验，在当时被认为是违反量子力学的一个显著例证。

量子力学的发展不仅深刻影响了我们的理解和应用技术，也激发了许多新的科学发现和技术突破，例如半导体技术和激光技术的诞生。然而，量子力学仍然充满了未知和挑战，尽管科学家们不断努力寻找更深层次的理解和应用。


### 数据集处理

法律数据集链接：https://huggingface.co/datasets/ShengbinYue/DISC-Law-SFT

In [ ]:
from datasets import load_dataset

# ds = load_dataset("BAAI/COIG-PC-Lite")

# 读取法律 QA 问答对数据集 
ds = load_dataset(
    "json",
    data_files="C:/Users/Jason/.cache/huggingface/hub/datasets--ShengbinYue--DISC-Law-SFT/snapshots/fb12cf02809a85724f7e36977529b1d4b5f9f920/DISC-Law-SFT-Pair-QA-released.jsonl"
)

In [54]:
print(ds)

print(ds["train"])
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'input', 'output'],
        num_rows: 79692
    })
})
Dataset({
    features: ['id', 'input', 'output'],
    num_rows: 79692
})
{'id': 'legal_question_answering_0', 'input': '违章停车与违法停车是否有区别？', 'output': '对违反道路交通安全法律、法规关于机动车停放、临时停车规定的，可以指出违法行为，并予以口头警告，令其立即驶离。机动车驾驶人不在现场或者虽在现场但拒绝立即驶离，妨碍其他车辆、行人通行的处二十元以上二百元以下罚款。现在人们大多是称作违法停车，因此在法律责任上也会更多一些，不要以为违反交通规章制度问题不大，不要认为违法停车是罚款而已。'}


发现数据集中**只有一个训练集**，需要进行人工划分

划分方式：

训练/验证/测试：3000/500/500

其实不进行划分也没事，因为这样的小模型只能起到知识注入的功能，并不能推理。可以只使用训练集，对比微调前后的模型回答差异，验证是否学到相关法律知识。

In [22]:
data = ds["train"].shuffle(seed=42)

train_ds = data.select(range(0, 3000))
valid_ds = data.select(range(3000, 3500))
test_ds = data.select(range(3500, 4000))

print(len(train_ds))
print(len(valid_ds))
print(len(test_ds))
print(train_ds.column_names)

3000
500
500
['id', 'input', 'output']


在训练的时候，输入给模型的应该是一个纯文本的字符串列表，字符串本身就应该带有对话模板中的文字内容

所以，在按列提取完数据集中的对话内容后，还需要转换成标准对话形式

In [23]:


def to_messages(example):
    """
    将问答对转换为对话形式
    """
    return {
        "messages": [
            {"role": "user", "content": example["input"]},
            {"role": "assistant", "content": example["output"]},
        ]
    }

def format_with_template(example):
    """
    转换为具有对话模板的字符串
    """
    text = tokenizer.apply_chat_template(
        example["messages"], 
        tokenize=False ,
        add_generation_prompt=False ,
    ) 
    
    return {"text": text}

def length_filter(example , max_tokens=8192):
    """
    对数据集中模型回复部分 token 数量进行过滤
    """
    ids = tokenizer(example["text"], truncation=False).input_ids

    return len(ids) <= max_tokens 


train_ds = train_ds.map(
    to_messages,
    remove_columns=train_ds.column_names,
)
train_ds = train_ds.map(
    format_with_template,
    remove_columns=train_ds.column_names,
)
train_ds = train_ds.filter(lambda x: length_filter(x, max_tokens=8192))

valid_ds = valid_ds.map(
    to_messages,
    remove_columns=valid_ds.column_names,
)

test_ds = test_ds.map(
    to_messages,
    remove_columns=test_ds.column_names,
)
print(train_ds[0])
print(train_ds[0]["text"])

Filter:   0%|          | 0/3000 [00:00<?, ? examples/s]

{'text': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n请根据所给的具体罪名，给出其对应法条\n合同诈骗罪<|im_end|>\n<|im_start|>assistant\n[刑法条文]\n第二百二十四条有下列情形之一，以非法占有为目的，在签订、履行合同过程中，骗取对方当事人财物，数额较大的，处三年以下有期徒刑或者拘役，并处或者单处罚金;数额巨大或者有其他严重情节的，处三年以上十年以下有期徒刑，并处罚金;数额特别巨大或者有其他特别严重情节的，处十年以上有期徒刑或者无期徒刑，并处罚金或者没收财产：\n(一)以虚构的单位或者冒用他人名义签订合同的;\n(二)以伪造、变造、作废的票据或者其他虚假的产权证明作担保的;\n(三)没有实际履行能力，以先履行小额合同或者部分履行合同的方法，诱骗对方当事人继续签订和履行合同的;\n(四)收受对方当事人给付的货物、货款、预付款或者担保财产后逃匿的;\n(五)以其他方法骗取对方当事人财物的。\n第二百三十一条单位犯本节第二百二十一条至第二百三十条规定之罪的，对单位判处罚金，并对其直接负责的主管人员和其他直法接责任人员，依照本节各该条的规定处罚。<|im_end|>\n'}
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
请根据所给的具体罪名，给出其对应法条
合同诈骗罪<|im_end|>
<|im_start|>assistant
[刑法条文]
第二百二十四条有下列情形之一，以非法占有为目的，在签订、履行合同过程中，骗取对方当事人财物，数额较大的，处三年以下有期徒刑或者拘役，并处或者单处罚金;数额巨大或者有其他严重情节的，处三年以上十年以下有期徒刑，并处罚金;数额特别巨大或者有其他特别严重情节的，处十年以上有期徒刑或者无期徒刑，并处罚金或者没收财产：
(一)以虚构的单位或者冒用他人名义签订合同的;
(二)以伪造、变造、作废的票据或者其他虚假的产权证明作担保的;

<div style="page-break-after: always;"></div>

# 2026/8/20

### 实战：模型微调三步走

**1. 模型加载** 

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
new_model_name = "Qwen2.5-Law-SFT-v1"

print(f"Loading {model_name}...")
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    dtype=torch.bfloat16, 
    device_map="cuda"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # 使用 eos_token 填充补齐短文本的占位符
tokenizer.padding_side = "right"    # 在文本右侧补齐

print(f"Model loaded! Parameters: {model.num_parameters():,}")

Loading Qwen/Qwen2.5-0.5B-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded! Parameters: 494,032,768


**2. dataset 构建**

In [2]:
print("=== DATASET 准备中 ===\n")

ds = load_dataset(
    "json",
    data_files="C:/Users/Jason/.cache/huggingface/hub/datasets--ShengbinYue--DISC-Law-SFT/snapshots/fb12cf02809a85724f7e36977529b1d4b5f9f920/DISC-Law-SFT-Pair-QA-released.jsonl"
)

# 取出 3000 条用于训练
train_dataset = ds["train"].select(range(0, 3000))
print(train_dataset)

def to_prompt_completion(example):
    return {
        "prompt": [
            {"role": "user", "content": example["input"].strip()}
        ],
        "completion": [
            {"role": "assistant", "content": example["output"].strip()}
        ],
    }

train_dataset = train_dataset.map(to_prompt_completion, remove_columns=train_dataset.column_names)

print(train_dataset[0])
print(train_dataset)



=== DATASET 准备中 ===

Dataset({
    features: ['id', 'input', 'output'],
    num_rows: 3000
})
{'prompt': [{'role': 'user', 'content': '违章停车与违法停车是否有区别？'}], 'completion': [{'role': 'assistant', 'content': '对违反道路交通安全法律、法规关于机动车停放、临时停车规定的，可以指出违法行为，并予以口头警告，令其立即驶离。机动车驾驶人不在现场或者虽在现场但拒绝立即驶离，妨碍其他车辆、行人通行的处二十元以上二百元以下罚款。现在人们大多是称作违法停车，因此在法律责任上也会更多一些，不要以为违反交通规章制度问题不大，不要认为违法停车是罚款而已。'}]}
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 3000
})


**3. 训练参数配置**

In [ ]:
training_config = SFTConfig(
    # 模型和数据
    output_dir=f"sft_models/{new_model_name}",
    max_length=8192,
    completion_only_loss=True,

    # 训练超参数
    per_device_train_batch_size=3,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    num_train_epochs=1,  # Start with 1 epoch
    gradient_checkpointing=True,
    bf16=True,
    fp16=False,

    # 优化器
    warmup_steps=50,
    weight_decay=0.01,
    optim="adamw_torch",

    # 日志及保存
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,

    # 实验记录
    report_to="trackio",
    run_name=f"{new_model_name}-training",
)

print("训练配置完成!")
print(f"有效批次大小: {training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps}")

训练配置完成!
有效批次大小: 6


初始化训练器

In [4]:
# 训练器初始化
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=None,
    args=training_config,
)

检查数据格式

In [5]:
print(tokenizer.eos_token)
print(tokenizer.eos_token_id)
print(repr(train_dataset[0]))

<|im_end|>
151645
{'prompt': [{'role': 'user', 'content': '违章停车与违法停车是否有区别？'}], 'completion': [{'role': 'assistant', 'content': '对违反道路交通安全法律、法规关于机动车停放、临时停车规定的，可以指出违法行为，并予以口头警告，令其立即驶离。机动车驾驶人不在现场或者虽在现场但拒绝立即驶离，妨碍其他车辆、行人通行的处二十元以上二百元以下罚款。现在人们大多是称作违法停车，因此在法律责任上也会更多一些，不要以为违反交通规章制度问题不大，不要认为违法停车是罚款而已。'}]}


验证是否将模型回复之前的 token 都进行 mask，下面的空格部分代表 mask 部分

In [6]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for  x in trainer.train_dataset [100]["labels"]]).replace(tokenizer. pad_token , " ")

'                                       根据《国有土地上房屋征收与补偿条例》第23条的规定，对因征收房屋造成停产停业损失的补偿，根据房屋被征收前的效益、停产停业期限等因素确定，具体办法由省、自治区、直辖市制定。对此，各地方政府对停产停业损失的补偿对象作出了具体规定。例如，根据《重庆市国有土地上房屋征收停产停业损失补偿办法（暂行）》（已失效）第4条的规定，停产停业损失的补偿对象应当符合以下条件：首先，登记为非住宅用途，实际产生经济效益的房屋均可获得停产停业补偿。其次，对于登记为住宅用途，但实际用于经营的房屋，在符合下列条件的情况下，也可以获得停产停业补偿：第一，被征收房屋具有房屋权属证明或者经有关部门认定为合法建筑；第二，有合法、有效的营业执照或者其他相关生产经营行政许可手续，且营业执照或者其他相关生产经营行政许可手续上载明的营业场所为被征收房屋；第三，已办理税务登记并具有税负核定凭证；第四，因征收房屋造成了停产停业损失。\xa0法律依据：《国有土地上房屋征收与补偿条例》第二十三条\u3000对因征收房屋造成停产停业损失的补偿，根据房屋被征收前的效益、停产停业期限等因素确定。具体办法由省、自治区、直辖市制定。 \n'

In [6]:
batch = next(iter(trainer.get_train_dataloader()))

labels = batch["labels"]

print("labels shape:", labels.shape)
print("有效 label 数量:", (labels != -100).sum().item())
print("总 label 数量:", labels.numel())


for i in range(batch["input_ids"].shape[0]):
    valid_ids = batch["input_ids"][i][batch["labels"][i] != -100]

    print(f"样本 {i} 有效目标文本：")
    print(tokenizer.decode(valid_ids, skip_special_tokens=False))

labels shape: torch.Size([3, 230])
有效 label 数量: 342
总 label 数量: 690
样本 0 有效目标文本：
《工伤保险条例》第十六条：职工符合本条例第十四条、第十五条的规定，但是有下列情形之一的，不得认定为工伤或者视同工伤：（一）故意犯罪的；（二）醉酒或者吸毒的；（三）自残或者自杀的。<|im_end|>

样本 1 有效目标文本：
当事人请求返还按照习俗给付的彩礼的，如果查明属于以下情形，人民法院应当予以支持：(一)双方未办理结婚登记手续;(二)双方办理结婚登记手续但确未共同生活;(三)婚前给付并导致给付人生活困难。适用前款第二项、第三项的规定，应当以双方离婚为条件。人民法院适用普通程序审理的案件，应当在立案之日起六个月内审结。有特殊情况需要延长的，由本院院长批准，可以延长六个月;还需要延长的，报请上级人民法院批准。人民法院适用简易程序审理案件，应当在立案之日起三个月内审结。去法院起诉离婚，需要证明夫妻之间感情分离，立案之后大概会在一个月内开庭，诉讼离婚属于民事案件，可以适用于普通程序审理，也可以适用于简易程序审理，若适用普通程序审理，需要六个月的时间，若是用于简易程序，需要三个月的时间。<|im_end|>

样本 2 有效目标文本：
独生子直接继承房产可以，不过前提是被继承人没有留有遗赠或者遗赠扶养协议。如果有遗赠的的，由其约定的继承人继承，如果有遗赠扶养协议的，则按照协议约定的方式发生继承。但是要注意，即使有遗赠，但是该独生子是缺乏劳动能力又没有生活来源的，应当为其保留必要的份额。<|im_end|>



### 开始训练

In [7]:
print("\n=== 开始训练 ===")
trainer.train()

# 保存模型
trainer.save_model()
print(f"Model saved to {training_config.output_dir}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.



=== 开始训练 ===
* Trackio project initialized: huggingface
* Trackio metrics logged to: C:\Users\Jason\.cache\huggingface\trackio
* psutil detected, enabling automatic CPU/system metrics logging
* Resumed existing run: Qwen2.5-Law-SFT-v1-training


Step,Training Loss
10,1.976652
20,1.993444
30,1.801110
40,1.719763
50,1.910061
60,1.804111
70,1.737039
80,1.778384
90,1.843977
100,1.754709


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

* Run finished. Uploading logs to Trackio (please wait...)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./Qwen2.5-Law-SFT-v1


### 模型效果对比

In [1]:
# 对比基础模型和微调模型
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

# Notebook 当前目录为 llm_study/week4/notes
base_path = "Qwen/Qwen2.5-0.5B-Instruct"
finetuned_path = Path("sft_models/Qwen2.5-Law-SFT-v2")
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("设备：", device)
if device == "cuda":
    print("显卡：", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(
    finetuned_path, local_files_only=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_path,
    dtype=dtype,
).to(device).eval()
finetuned_model = AutoModelForCausalLM.from_pretrained(
    finetuned_path,
    dtype=dtype,
    local_files_only=True,
).to(device).eval()
print("基础模型和微调模型加载完成")

def generate_answer(model, question, max_new_tokens=768):
    messages = [
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

test_questions = [
    "违章停车与违法停车是否有区别？",
    "哪些情形不得被认定为工伤？",
    "征收哪类房屋可以获得停产停业补偿？",
]

for question in test_questions:
    print("问题：", question)
    print("基础模型：")
    print(generate_answer(base_model, question))
    print("微调模型：")
    print(generate_answer(finetuned_model, question))
    print("-" * 60)

设备： cuda
显卡： NVIDIA GeForce RTX 5060 Ti


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

基础模型和微调模型加载完成
问题： 违章停车与违法停车是否有区别？
基础模型：
违章停车和违法停车是两个不同的概念，它们在法律上有着明显的区别。

1. **违章停车**：这是指违反交通法规、道路使用规定或安全规则的行为。例如，闯红灯、不按规定车道行驶、超速行驶等都是违章停车的表现形式。这些行为不仅违反了交通规则，还可能对其他车辆和行人造成安全隐患，甚至构成犯罪。

2. **违法停车**：这是指违反法律法规的行为，包括但不限于：
   - 违反道路交通管理法规。
   - 驾驶不符合国家规定的机动车。
   - 未取得驾驶资格的人员驾驶机动车。
   - 使用伪造、变造或者盗用他人机动车号牌、驾驶证、行驶证、检验合格标志、保险标志等证件、证明。
   - 在道路上进行妨碍安全驾驶的活动（如鸣笛、抛洒物品）。
   - 其他违反道路交通管理法规的行为。

两者的主要区别在于：
- **目的不同**：违章停车主要是为了遵守交通法规，保障自身及他人的安全；而违法停车则有多种目的，可能是出于个人利益、经济利益或其他正当理由。
- **后果不同**：违章停车可能导致交通事故，影响行车安全；违法停车可能会导致罚款、扣分、吊销驾照等处罚，严重时还会面临刑事追究。
- **法律责任不同**：违章停车通常由交警部门处理，责任一般由当事人承担；违法停车的责任主体通常是驾驶员本人，如果发生事故，则由保险公司负责赔偿。

因此，在日常生活中，我们应当严格遵守交通法规，文明出行，避免违章停车，共同维护良好的交通秩序和社会环境。
微调模型：
违章停车和违法停车是两种不同的行为，它们之间存在一定的区别。违章停车是指在道路上违反交通规则的行为，如闯红灯、超速行驶等；而违法停车则是指在道路上违反法律规定的行为，如酒驾、醉驾等。
违章停车通常会导致罚款、扣分等处罚措施，而违法停车则可能面临更严重的后果，例如被吊销驾照或被追究刑事责任等。
因此，在日常生活中，我们应该遵守交通法规，避免违章停车，以保障自己的安全和权益。同时，我们也应该尊重他人的权利，不进行违法行为，共同维护良好的道路交通环境。
------------------------------------------------------------
问题： 哪些情形不得被认定为工伤？
基础模型：
根据《中华人民共和国社会保险法》和《国务

## 全量 SFT 实战踩坑复盘

本次使用 `Qwen/Qwen2.5-0.5B-Instruct` 和法律问答数据进行全量微调。以下问题按实际训练流程整理。

### 1. CPU 版 PyTorch 无法使用显卡

- **现象**：`torch.__version__` 显示 `+cpu`，`torch.cuda.is_available()` 为 `False`。
- **原因**：安装的是不包含 CUDA Runtime 的 PyTorch 包。`nvidia-smi` 显示的 CUDA 版本代表驱动支持上限，不等于 PyTorch 已经支持 CUDA。
- **解决**：安装 CUDA 版 PyTorch，并通过 `torch.version.cuda`、`torch.cuda.is_available()` 与 `torch.cuda.get_device_name(0)` 确认 GPU 可用。

### 2. 数据需要和训练目标匹配

- **现象**：原始数据只有 `id`、`input`、`output`，不能直接表达“用户提问、模型回答”的监督边界。
- **原因**：SFTTrainer 需要知道完整对话或 prompt 与 completion 的分界，才能构造正确的 labels。
- **解决**：本次最终采用 conversational prompt-completion 格式：

```python
{
    "prompt": [{"role": "user", "content": input}],
    "completion": [{"role": "assistant", "content": output}],
}
```

配合 `completion_only_loss=True`，模型只学习回答部分。训练、验证和测试数据必须使用同一种结构。

### 3. `text`、`messages` 与 prompt-completion 不能混用

- **现象**：已经手动用 `apply_chat_template()` 生成 `text` 后，再设置 `assistant_only_loss=True` 会报“数据不是 conversational”的错误。
- **原因**：`text` 只是普通字符串，Trainer 无法从中可靠识别 user 和 assistant 的边界；`assistant_only_loss` 只支持结构化对话数据。
- **解决**：三种路线只能选一种并保持一致：普通 `text` 使用整段 loss；`messages` 使用 `assistant_only_loss`；`prompt/completion` 使用 `completion_only_loss`。本次使用最后一种。

### 4. Qwen Chat Template 不支持 assistant generation mask

- **现象**：设置 `assistant_only_loss=True` 后，解码有效 labels 仍然包含 system 和 user。
- **原因**：Qwen 的 `chat_template.jinja` 没有 `{% generation %}` / `{% endgeneration %}` 标记，TRL 无法从 `messages` 自动建立 assistant token mask。
- **解决**：不依赖 `assistant_only_loss`，改为 prompt-completion 数据格式和 `completion_only_loss=True`。检查 labels 时，解码出的有效 token 应只包含 assistant 回答与结束标记。

### 5. 手动套模板会造成 EOS 重复

- **现象**：样本末尾出现两个 `<|im_end|>`。
- **原因**：`apply_chat_template()` 已经为 assistant 回答加入结束标记，SFTTrainer 又自动追加 EOS。
- **解决**：使用结构化 prompt-completion，让 Trainer 统一套模板；不要提前生成 `text`。如果必须使用 `text`，需要关闭 Trainer 的额外 EOS，并检查每条文本的结束标记数量。

### 6. FP16 全量更新导致 loss 归零、模型只输出感叹号

- **现象**：开始时 loss 正常，随后大量 step 显示 `0.000000`；微调模型生成连续感叹号。
- **原因**：模型以 FP16 权重直接进行全量参数更新，但训练配置没有正确启用混合精度。全量 FP16 更新数值范围和精度不足，容易发生下溢、溢出或权重退化。
- **解决**：重新从基础模型开始训练，使用 BF16：

```python
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16
).to("cuda")

SFTConfig(
    bf16=True, fp16=False,
    learning_rate=1e-5,
    gradient_checkpointing=True,
)
```

不能继续使用已经退化的 checkpoint。当前 loss 能持续下降，说明新的数值配置已经正常工作。